# FedGNN v7 runner

Runtime: keep the default **CPU** runtime. On 2 CPU cores these small graphs train about as fast as on a T4, results are bit-identical between runs, and CPU sessions have no GPU quota.

1. Set `JOB` in cell 1 (one job per Colab account at a time; job table in cell 5).
2. Run cells 1, 2 and 4. Cell 3 is needed only once, on the first account.
3. If Colab disconnects, run cells 1, 2 and 4 again: the job resumes where it stopped.


In [ ]:
# Cell 1 - settings
JOB = "A1"                                      # change for each account
DRIVE_FOLDER = "/content/drive/MyDrive/FedGNN_v7"

In [ ]:
# Cell 2 - setup (code, library, data)
from google.colab import drive
drive.mount("/content/drive")

import os, shutil, subprocess, zipfile, glob
assert os.path.isdir(DRIVE_FOLDER), f"Folder not found: {DRIVE_FOLDER}"
code_zip = os.path.join(DRIVE_FOLDER, "fedgnn_v7_code.zip")
assert os.path.isfile(code_zip), f"Missing {code_zip}"
shutil.rmtree("/content/fedgnn_v7", ignore_errors=True)
with zipfile.ZipFile(code_zip) as z:
    z.extractall("/content")
assert os.path.isfile("/content/fedgnn_v7/main.py"), "zip layout unexpected"

subprocess.run(["pip", "install", "-q", "torch_geometric"], check=True)

os.makedirs("/content/data", exist_ok=True)
for name in ["elliptic_txs_features.csv", "elliptic_txs_classes.csv", "elliptic_txs_edgelist.csv"]:
    src = os.path.join(DRIVE_FOLDER, "data", name)
    dst = os.path.join("/content/data", name)
    assert os.path.isfile(src), f"Missing data file in Drive: {src}"
    if not os.path.isfile(dst) or os.path.getsize(dst) != os.path.getsize(src):
        print("copying", name); shutil.copy(src, dst)

import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU runtime)")
os.chdir("/content/fedgnn_v7")

In [ ]:
# Cell 3 - ONE TIME ONLY: determinism + reproduction check (~6 min)
!python tools/determinism_check.py --work /content/detcheck


In [ ]:
# Cell 4 - run the job (resumes automatically after a disconnect)
!python tools/run_job.py {JOB} --results-root {DRIVE_FOLDER}/results

In [ ]:
# Cell 5 - progress of all jobs (any account can run this) and the job table
!python tools/merge_results.py {DRIVE_FOLDER}/results
!python tools/jobs.py

In [ ]:
# Cell 6 - when ALL jobs are finished: build the zip to send back
!python tools/export_results.py {DRIVE_FOLDER}/results
